In [1]:
%load_ext autoreload
%autoreload 2

from app.logger import *
import json5,json
import fitz #type: ignore
import pandas as pd
import numpy as np

from app.sidkim.fund_data import *
from app.utils import *
from app.parse_table import *
with open("config\\sidkim\\sid_params.json5","r+") as file:
    params = json5.load(file)

# | Tool      | Page Indexing | Example                  |
# | --------- | ------------- | ------------------------ |
# | `fitz`    | **0-based**   | `doc[0]` → first page    |
# | `camelot` | **1-based**   | `pages="1"` → first page |


In [4]:
path = r"C:\Users\kaustubh.keny\Downloads\98_17834_Dec-2025_1765166712_KIM.pdf"
object = JioBlackRockSIDKIM(amc_id="98",path=path)
extracted_text = object.parse_KIM_data(pages="3", instrument_count=2)
data = object.refine_data(extracted_text)
dfs = object.merge_and_select_data(data=data,sid_or_kim="kim",special_func=False)

[ROW START]: [2, 3, 4]
Function Running: refine_data
Function Running: merge_and_select_data


In [ ]:
path = r"C:\Users\kaustubh.keny\Downloads\96_17682_Oct-2025_1761884629_KIM.pdf"
object = AngelOneSIDKIM(amc_id="96",path=path)
extracted_text = object.parse_KIM_data(pages="2", instrument_count=4)
data = object.refine_data(extracted_text)
dfs = object.merge_and_select_data(data=data,sid_or_kim="kim",special_func=False)

In [5]:
with open("temp.json","w+") as file:
    json.dump(extracted_text,file)
with open("data.json","w+") as file:
    json.dump(data,file)
with open("dfs.json","w+") as file:
    json.dump(dfs,file)

In [ ]:
path = r"C:\Users\kaustubh.keny\Downloads\99_17752_Nov-2025_1763359711_SID.pdf"
object = CapitalMindSIDKIM(amc_id="99", path=path)
temp_dict = {}
temp_dict.update(object.parse_page_zero("1"))
temp_dict.update(object.parse_scheme_table_data("5-12"))
temp_dict.update(object.parse_fund_manager_info("23"))

In [2]:
path = r"C:\Users\kaustubh.keny\Downloads\98_17833_Dec-2025_1765166683_SID.pdf"
object = JioBlackRockSIDKIM(amc_id="98", path=path)
temp_dict = {}
temp_dict.update(object.parse_page_zero("1"))
temp_dict.update(object.parse_scheme_table_data("4-12"))
temp_dict.update(object.parse_fund_manager_info("20"))

Function Running: parse_page_zero
Function Running: parse_scheme_table_data
[COL START]: [2, 3]
Function Running: parse_fund_manager_info
[ROW START]: [1]


In [3]:
data = object.refine_data(temp_dict)
dfs = object.merge_and_select_data(data,sid_or_kim="sid",special_func=False)
with open("temp.json","w+") as file:
    json.dump(temp_dict,file)
with open("data.json","w+") as file:
    json.dump(data,file)
with open("dfs.json","w+") as file:
    json.dump(dfs,file)

Function Running: refine_data
Function Running: merge_and_select_data


In [4]:
from app.sqlconnect import *

db_config =  {
        "host": "172.22.225.155",
        "port": 3306,
        "user": "cog_mf",
        "password": "bnYwFChjLAV2Z%9E",
        "database": "cog_mf"
    }

conn = establish_connection(db_config)
path = r"60_30-Nov-25_FS.json"
json_to_cog_db(path,db_config)

True

In [ ]:
import json

with open("registry.json", "r", encoding="utf-8") as f:
    data = json.load(f)

def strip_symbols(text):
    return text.replace("₹", "") if isinstance(text, str) else text

for amc in data.get("amc_registry", {}).values():
    for k, v in amc.items():
        amc[k] = strip_symbols(v)

with open("registry_clean.json", "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2)